In [19]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

In [22]:
#a)

X_fantoma = np.array([
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
[0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
[0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
[0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)
x_fantoma = X_fantoma.reshape(-1)


def mascara_rayo(n, direccion, s):
    res =  np.zeros((n, n))
    for i in range(n) :
        for j in range(n):
            if (direccion[0]*i + direccion[1]*j) == s:
                res[i,j] =1
    return res

def matriz_proyeccion(n, direcciones):
    filas_A = []
    for direccion in direcciones:
        valoresdes=[]
        for i in range(n):
            for j in range(n):
                s = direccion[0]*i + direccion[1]*j
                valoresdes.append(s)
    
        ssinrepetir = []

        for k in valoresdes:
            if k not in ssinrepetir:
                ssinrepetir.append(k)        

        ssinrepetir.sort()

        for s in ssinrepetir:
            fila_s =mascara_rayo(n, direccion, s)
            filas_A.append(fila_s.reshape(-1))
           
    A=np.array(filas_A)
    return A

d7 = [(1,0), (0,1), (1,1), (1, -1), (2,1),(1,2),(3,1)]

A7= matriz_proyeccion(8, d7)

def generar_mediciones(A, x):
    return A @ x

b7 = generar_mediciones(A7, x_fantoma)   

np.random.seed(42)
z = np.random.randn(len(b7))
escala = np.sqrt(np.mean(b7**2))

sigma = 0.06
eta = sigma * escala * z
b_sigma = b7 + eta

def reconstruir(A, b):
    x_hat, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    return x_hat

x_hat_sigma = reconstruir(A7, b_sigma)

def residuo(Ak, bk, xk_hat, x_fantoma):
    resk = bk - Ak@xk_hat
    norma_resk = np.linalg.norm(resk)
    error_relativo = np.linalg.norm(xk_hat - x_fantoma) / np.linalg.norm(x_fantoma)
    return norma_resk, error_relativo

res_sigma= b_sigma - A7 @ x_hat_sigma


print(np.linalg.norm(A7.T @res_sigma))





4.3641290461405304e-14


b)
Como A7^T@r_sigma es casi cero (4.3641290461405304e-14), el residuo es ortogonal a todas las columnas de A7. Entonces A7x es el vector de Col(A7) mas cercano a b_sigma, por lo que representa la proyeccion ortogonal de b_sigma sobre el Col(A7). 

c)
Las proyecciones tomográficas son las mediciones que obtiene el escáner a partir de los rayos que atraviesan la imagen. En cambio, la proyección ortogonal de mínimos cuadrados es una operación matemática que busca, dentro de Col(A7), el vector más cercano a las mediciones ruidosas. Aunque ambas se llaman proyección, representan conceptos distintos.